In [ ]:
!pip install -q -U transformers accelerate qwen-vl-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 14.0 MB/s eta 0:00:00


In [ ]:
from google.colab import drive, userdata
from huggingface_hub import login
drive.mount("/content/drive")
login(userdata.get("HF_TOKEN"))

Mounted at /content/drive


In [ ]:
PROMPT_ID = "caption_v1"
PROMPT = (
    "Write a figure caption for this medical image as it would appear in a radiology "
    "case report. State the imaging modality and anatomical region, and describe visible "
    "findings. Do not speculate about the diagnosis. Respond in one or two sentences."
)
MAX_NEW_TOKENS = 100
N_IMAGES = None

MODEL_NAME = "MedMO-4B-Next"

DRIVE = "/content/drive/MyDrive/vlm-caption-bench"
SUBSET_FILE = f"{DRIVE}/subset_2000.csv"
OUT_DIR = f"{DRIVE}/predictions"

In [ ]:
import os, zipfile
import pandas as pd
from pathlib import Path
from PIL import Image

os.makedirs(OUT_DIR, exist_ok=True)
if not os.path.isdir("test"):
    with zipfile.ZipFile(f"{DRIVE}/test.zip") as z:
        z.extractall("test")

ids = pd.read_csv(SUBSET_FILE)["ID"].astype(str).tolist()
if N_IMAGES:
    ids = ids[:N_IMAGES]
by_stem = {p.stem: p for p in Path("test").rglob("*.jpg")}
missing = [i for i in ids if i not in by_stem]
assert not missing, f"немає файлів для {len(missing)} ID, напр. {missing[:3]}"
paths = [by_stem[i] for i in ids]
print(len(paths), "images")

2000 images


In [ ]:
import torch
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

model_id = "MBZUAI/MedMO-4B-Next"
dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
model = Qwen3VLForConditionalGeneration.from_pretrained(model_id, torch_dtype=dtype, device_map="auto")
processor = AutoProcessor.from_pretrained(model_id)

config.json:   0%|          | 0.00/1.60k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/64.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/714 [00:00<?, ?it/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

In [ ]:
def caption(image, prompt=PROMPT):
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": prompt},
    ]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, videos=video_inputs,
                       padding=True, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    return processor.decode(ids[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

In [ ]:
for p in paths[:3]:
    print(p.name, "->", caption(Image.open(p).convert("RGB")), "\n")

/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(


ROCOv2_2023_test_005382.jpg -> Coronal T2-weighted MRCP image showing normal appearance of the gallbladder and common bile duct. No stones or sludge are seen. 

ROCOv2_2023_test_004808.jpg -> Thoracic lateral radiograph showing a thoracic vertebral body height of 10.5 mm and a thoracic vertebral body width of 12.5 mm. 

ROCOv2_2023_test_002741.jpg -> Color Doppler ultrasonography showing the intima-media thickness (IMT) of the carotid artery. 



In [ ]:
import json, os, time, torch

OUT = Path(OUT_DIR) / f"{MODEL_NAME}__{PROMPT_ID}.jsonl"

def read_done(path):
    ids, good = set(), []
    if path.exists():
        for l in path.open(encoding="utf-8"):
            try:
                r = json.loads(l)
            except json.JSONDecodeError:
                continue
            ids.add(r["image_id"])
            good.append(l if l.endswith("\n") else l + "\n")
        path.write_text("".join(good), encoding="utf-8")
    return ids

done = read_done(OUT)
print(len(paths), "images,", len(done), "already done")

failed = []
with OUT.open("a", encoding="utf-8") as f:
    for i, p in enumerate(paths, 1):
        image_id = p.stem
        if image_id in done:
            continue
        try:
            cap = caption(Image.open(p).convert("RGB"))
        except torch.cuda.OutOfMemoryError:
            print(f"OOM on {p.name} - пропущено, буде повторено при наступному запуску")
            failed.append(p.name)
            torch.cuda.empty_cache()
            continue
        rec = {
            "image_id": image_id,
            "caption": cap,
            "model": MODEL_NAME,
            "prompt_id": PROMPT_ID,
            "max_new_tokens": MAX_NEW_TOKENS,
            "do_sample": False
        }
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        f.flush()
        os.fsync(f.fileno())
        torch.cuda.empty_cache()
        if i % 50 == 0:
            print(f"{i}/{len(paths)} done")
print("failed:", failed)

2000 images, 2 already done
50/2000 done
100/2000 done
150/2000 done
200/2000 done
250/2000 done
300/2000 done
350/2000 done
400/2000 done
450/2000 done
500/2000 done
550/2000 done
600/2000 done
650/2000 done
700/2000 done
750/2000 done
800/2000 done
850/2000 done
900/2000 done
950/2000 done
1000/2000 done
1050/2000 done
1100/2000 done
1150/2000 done
1200/2000 done
1250/2000 done
1300/2000 done
1350/2000 done
1400/2000 done
1450/2000 done
1500/2000 done
1550/2000 done
1600/2000 done
1650/2000 done
1700/2000 done
1750/2000 done
1800/2000 done
1850/2000 done
1900/2000 done
1950/2000 done
2000/2000 done
failed: []
